# Examen BQ — Feature Engineering PyPI

Este notebook:
1. Carga credenciales y nombres de tabla desde `.env` usando `python-dotenv`
2. Lee el query `scripts/BQ/examen.sql` (que contiene el placeholder `{BQ_TABLE}`)
3. Sustituye el placeholder con el valor real del `.env`
4. Ejecuta el query en BigQuery y retorna un `DataFrame` de pandas

## 2. Carga de variables de entorno

In [1]:
import os
from pathlib import Path
from dotenv import load_dotenv

# Subimos dos niveles desde notebooks/ hasta la raíz del proyecto
ROOT = Path("__file__").resolve().parent.parent
dotenv_path = ROOT / ".env"

load_dotenv(dotenv_path=dotenv_path, override=True)

# ── Variables requeridas en .env ──────────────────────────────────────────
# BQ_PROJECT  → GCP Project ID  (ej. bi-anahuac-ene-mar-2026)
# BQ_TABLE    → Tabla completa   (ej. bi-anahuac-ene-mar-2026.mock_exam.agg_pypi_sample)
# GOOGLE_APPLICATION_CREDENTIALS → ruta al JSON de la cuenta de servicio (opcional si usas ADC)

BQ_PROJECT = os.environ["BQ_PROJECT"]
BQ_TABLE   = os.environ["BQ_TABLE"]

## 3. Lectura del SQL y sustitución de placeholders

In [ ]:
SQL_PATH = ROOT / "scripts" / "BQ" / "examen.sql"

raw_sql = SQL_PATH.read_text(encoding="utf-8")

# str.format_map lanza KeyError si falta algún placeholder → falla rápido y explícito
query = raw_sql.format_map({"BQ_TABLE": BQ_TABLE})

print("✅ Query listo.")

✅ Query listo. Primeras 300 chars:


## 4. Conexión a BigQuery y ejecución del query

In [3]:
from google.cloud import bigquery

client = bigquery.Client(project=BQ_PROJECT)

print("⏳ Ejecutando query en BigQuery...")
df = client.query(query).to_dataframe()
print(f"✅ Resultados: {df.shape[0]:,} filas × {df.shape[1]} columnas")

⏳ Ejecutando query en BigQuery...
✅ Resultados: 4,422 filas × 41 columnas


## 5. Vista previa del DataFrame

In [4]:
df.head(10)

,ts,project,downloads,x_avg_dwn_1_5,x_std_dwn_1_5,x_avg_growth_1_5,x_avg_dwn_6_10,x_std_dwn_6_10,x_avg_growth_6_10,x_avg_dwn_11_15,...,x_std_dwn_46_50,x_avg_growth_46_50,x_avg_dwn_51_55,x_std_dwn_51_55,x_avg_growth_51_55,x_avg_dwn_56_60,x_std_dwn_56_60,x_avg_growth_56_60,tgt,rn
0,2026-03-11 01:00:00+00:00,boto3,54696,29964.0,5248.470491,0.007850,26963.6,781.768700,0.030047,32021.6,...,14339.076529,-0.071536,73403.2,6818.381567,0.038948,70888.8,23732.776675,0.195657,542891,61
1,2026-03-11 01:00:00+00:00,packaging,34450,31054.2,1063.675561,-0.008384,31732.6,979.057353,0.022120,30315.4,...,2229.288564,0.010418,43286.4,4627.401031,-0.024287,41216.0,3527.871100,0.048782,363837,61
2,2026-03-11 01:00:00+00:00,urllib3,39078,25767.4,1610.349745,-0.014144,28437.2,2479.733595,0.034727,27053.4,...,1635.092566,-0.023127,42877.2,7599.715830,-0.063924,51302.4,5590.130795,0.036933,361043,61
3,2026-03-11 01:01:00+00:00,boto3,63001,33754.4,12415.852097,0.162481,28540.6,4071.353804,0.058578,31191.0,...,15273.919985,-0.016464,69874.2,11028.349908,-0.037241,77312.8,16452.931067,0.177101,527274,62
4,2026-03-11 01:01:00+00:00,packaging,37015,31747.6,1847.159522,0.024531,31606.2,1037.111228,-0.003034,30194.4,...,1761.217959,0.023147,40866.0,3866.727622,-0.054003,43622.6,4124.754696,0.058573,360232,62
5,2026-03-11 01:01:00+00:00,urllib3,43305,27960.4,6284.840754,0.090778,27703.4,1644.796431,-0.017954,27052.2,...,1714.091392,0.004233,40023.8,7045.406851,-0.061054,52299.6,4455.509657,0.027685,347529,62
6,2026-03-11 01:02:00+00:00,boto3,67209,39274.6,18144.600472,0.194774,30213.4,4926.964968,0.062561,28526.4,...,9672.469369,-0.009852,69331.8,10490.327626,0.024582,81181.0,13401.453746,0.095993,503157,63
7,2026-03-11 01:02:00+00:00,packaging,37636,33030.4,2821.831196,0.041888,31486.0,1126.633037,-0.002888,30177.4,...,1723.398474,0.002655,39078.2,2511.131558,-0.040688,45128.0,3221.692645,0.038033,355134,63
8,2026-03-11 01:02:00+00:00,urllib3,47760,31342.6,9135.147087,0.124641,27561.4,1737.516129,-0.000746,26898.0,...,1874.533355,-0.016061,36677.2,2765.193339,-0.071704,52985.6,3918.612293,0.019826,328055,63
9,2026-03-11 01:03:00+00:00,boto3,68915,47145.0,20354.110383,0.250748,30642.8,4512.483319,0.029755,26846.0,...,9576.358875,0.024787,65188.6,10295.081802,-0.022444,77311.8,9899.077265,-0.031427,470117,64


In [5]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 4422 entries, 0 to 4421
Data columns (total 41 columns):
 #   Column              Non-Null Count  Dtype              
---  ------              --------------  -----              
 0   ts                  4422 non-null   datetime64[us, UTC]
 1   project             4422 non-null   object             
 2   downloads           4422 non-null   Int64              
 3   x_avg_dwn_1_5       4422 non-null   float64            
 4   x_std_dwn_1_5       4422 non-null   float64            
 5   x_avg_growth_1_5    4422 non-null   float64            
 6   x_avg_dwn_6_10      4422 non-null   float64            
 7   x_std_dwn_6_10      4422 non-null   float64            
 8   x_avg_growth_6_10   4422 non-null   float64            
 9   x_avg_dwn_11_15     4422 non-null   float64            
 10  x_std_dwn_11_15     4422 non-null   float64            
 11  x_avg_growth_11_15  4422 non-null   float64            
 12  x_avg_dwn_16_20     4422 non-null 

In [6]:
df.describe()

,downloads,x_avg_dwn_1_5,x_std_dwn_1_5,x_avg_growth_1_5,x_avg_dwn_6_10,x_std_dwn_6_10,x_avg_growth_6_10,x_avg_dwn_11_15,x_std_dwn_11_15,x_avg_growth_11_15,...,x_std_dwn_46_50,x_avg_growth_46_50,x_avg_dwn_51_55,x_std_dwn_51_55,x_avg_growth_51_55,x_avg_dwn_56_60,x_std_dwn_56_60,x_avg_growth_56_60,tgt,rn
count,4422.0,4422.000000,4422.000000,4422.000000,4422.000000,4422.000000,4422.000000,4422.000000,4422.000000,4422.000000,...,4422.000000,4422.000000,4422.000000,4422.000000,4422.000000,4422.000000,4422.000000,4422.000000,4422.0,4422.0
mean,41463.921755,41447.919448,4810.470465,0.011598,41440.798598,4807.459830,0.011549,41435.719222,4811.150661,0.011527,...,4782.759783,0.011140,41505.459114,4792.979556,0.011059,41578.783401,4807.848150,0.011235,412969.166214,797.5
std,13853.593015,11878.442358,6375.405671,0.062426,11884.619253,6374.816506,0.062406,11889.577568,6373.820517,0.062399,...,6345.924546,0.061843,11893.064375,6348.843600,0.061870,11945.672839,6358.169860,0.061981,109348.113988,425.555171
min,21032.0,23621.000000,147.086709,-0.281126,23621.000000,147.086709,-0.281126,23621.000000,147.086709,-0.281126,...,147.086709,-0.281126,23621.000000,147.086709,-0.281126,23621.000000,147.086709,-0.281126,26.0,61.0
25%,33017.75,33708.200000,1497.504931,-0.014149,33700.750000,1497.177352,-0.014207,33700.750000,1498.802413,-0.014303,...,1497.177352,-0.014265,33767.500000,1500.254824,-0.014353,33820.000000,1500.254824,-0.014303,342564.5,429.0
50%,38947.5,39623.300000,2348.001284,0.000084,39623.300000,2345.934911,0.000061,39623.300000,2349.395235,0.000009,...,2340.046556,0.000055,39673.300000,2344.002643,0.000055,39761.500000,2345.934911,0.000084,399354.0,797.5
75%,45987.25,45957.450000,5809.937275,0.019426,45957.450000,5801.107606,0.019428,45957.450000,5809.937275,0.019428,...,5760.987653,0.019178,46019.350000,5801.107606,0.019161,46075.450000,5822.060365,0.019396,458341.0,1166.0
max,225920.0,149718.400000,74694.901519,0.499908,149718.400000,74694.901519,0.499908,149718.400000,74694.901519,0.499908,...,74694.901519,0.499908,149718.400000,74694.901519,0.499908,149718.400000,74694.901519,0.499908,1157597.0,1534.0
